<a href="https://colab.research.google.com/github/Hithavan31/BIS_PROGRAMS/blob/main/1BM24CS118_wireless%20sensor%20network%20optimization_%20Genetic%20Algorithm%20for%20Optimization%20Problem.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np

# ============================================================
# INPUT / PARAMETERS
# ============================================================

NUM_NODES = 10
NUM_CLUSTER_HEADS = 2

POP_SIZE = 20
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.1
MAX_GEN = 50

INITIAL_ENERGY = 100

# Generate fixed coordinates for wireless nodes
np.random.seed(42)

node_x = np.random.rand(NUM_NODES) * 100
node_y = np.random.rand(NUM_NODES) * 100


# ============================================================
# CALCULATE DISTANCE BETWEEN TWO NODES
# ============================================================

def calculate_distance(node1, node2):
    return np.sqrt(
        (node_x[node1] - node_x[node2]) ** 2 +
        (node_y[node1] - node_y[node2]) ** 2
    )


# ============================================================
# CREATE A NETWORK CONFIGURATION
# ============================================================

def create_network():

    # Randomly select Cluster Heads
    cluster_heads = random.sample(
        range(NUM_NODES),
        NUM_CLUSTER_HEADS
    )

    routes = []

    # Assign every node to its nearest Cluster Head
    for node in range(NUM_NODES):

        if node in cluster_heads:
            routes.append(node)

        else:
            nearest_head = min(
                cluster_heads,
                key=lambda ch: calculate_distance(node, ch)
            )

            routes.append(nearest_head)

    return [cluster_heads, routes]


# ============================================================
# FITNESS FUNCTION
# ============================================================

def fitness(network):

    cluster_heads, routes = network

    energy_consumption = 0

    # Calculate total communication energy
    for node in range(NUM_NODES):

        # Cluster Heads do not transmit to themselves
        if node not in cluster_heads:

            assigned_head = routes[node]

            distance = calculate_distance(
                node,
                assigned_head
            )

            # Energy is proportional to distance
            energy_consumption += distance + 1

    # Estimate network lifetime
    max_energy_used = max(
        energy_consumption,
        1
    )

    network_lifetime = (
        INITIAL_ENERGY / max_energy_used
    )

    # Lower fitness is better
    fitness_value = (
        energy_consumption +
        (1.0 / network_lifetime)
    )

    return fitness_value


# ============================================================
# SELECTION
# ============================================================

def select(population):

    # Sort population according to fitness
    # Lower fitness = better solution

    sorted_population = sorted(
        population,
        key=fitness
    )

    # Select the two best solutions
    return (
        sorted_population[0],
        sorted_population[1]
    )


# ============================================================
# CROSSOVER
# ============================================================

def crossover(parent1, parent2):

    # Select crossover point
    point = random.randint(
        1,
        NUM_CLUSTER_HEADS - 1
    )

    # Exchange Cluster Heads
    child1_heads = (
        parent1[0][:point] +
        parent2[0][point:]
    )

    child2_heads = (
        parent2[0][:point] +
        parent1[0][point:]
    )

    # Remove duplicate Cluster Heads
    child1_heads = list(
        dict.fromkeys(child1_heads)
    )

    child2_heads = list(
        dict.fromkeys(child2_heads)
    )

    # Fill missing Cluster Heads
    while len(child1_heads) < NUM_CLUSTER_HEADS:

        new_head = random.randrange(NUM_NODES)

        if new_head not in child1_heads:
            child1_heads.append(new_head)

    while len(child2_heads) < NUM_CLUSTER_HEADS:

        new_head = random.randrange(NUM_NODES)

        if new_head not in child2_heads:
            child2_heads.append(new_head)

    # Create children
    child1 = [child1_heads, []]
    child2 = [child2_heads, []]

    # Generate routes according to Cluster Heads
    for child in [child1, child2]:

        heads = child[0]
        routes = []

        for node in range(NUM_NODES):

            if node in heads:

                routes.append(node)

            else:

                nearest_head = min(
                    heads,
                    key=lambda ch:
                    calculate_distance(node, ch)
                )

                routes.append(nearest_head)

        child[1] = routes

    return child1, child2


# ============================================================
# MUTATION
# ============================================================

def mutate(network):

    cluster_heads, routes = network

    # --------------------------------------------------------
    # Mutate Cluster Head
    # --------------------------------------------------------

    if random.random() < MUTATION_RATE:

        index = random.randrange(
            NUM_CLUSTER_HEADS
        )

        new_head = random.randrange(
            NUM_NODES
        )

        # Make sure the new Cluster Head is unique
        while new_head in cluster_heads:

            new_head = random.randrange(
                NUM_NODES
            )

        cluster_heads[index] = new_head

    # --------------------------------------------------------
    # IMPORTANT:
    # Recalculate ALL routes after Cluster Head mutation
    # --------------------------------------------------------

    routes.clear()

    for node in range(NUM_NODES):

        if node in cluster_heads:

            routes.append(node)

        else:

            nearest_head = min(
                cluster_heads,
                key=lambda ch:
                calculate_distance(node, ch)
            )

            routes.append(nearest_head)

    # --------------------------------------------------------
    # Optional route mutation
    # --------------------------------------------------------

    if random.random() < MUTATION_RATE:

        node = random.randrange(
            NUM_NODES
        )

        if node not in cluster_heads:

            routes[node] = random.choice(
                cluster_heads
            )


# ============================================================
# GENETIC ALGORITHM
# ============================================================

def genetic_algorithm():

    # --------------------------------------------------------
    # INITIALIZE POPULATION
    # --------------------------------------------------------

    population = [
        create_network()
        for _ in range(POP_SIZE)
    ]

    best_network = None
    best_fitness = float("inf")

    # --------------------------------------------------------
    # GENERATION LOOP
    # --------------------------------------------------------

    for generation in range(
        1,
        MAX_GEN + 1
    ):

        # ----------------------------------------------------
        # EVALUATE POPULATION
        # ----------------------------------------------------

        for network in population:

            current_fitness = fitness(
                network
            )

            # Store best solution
            if current_fitness < best_fitness:

                best_fitness = current_fitness

                # Deep copy of network
                best_network = [
                    list(network[0]),
                    list(network[1])
                ]

        # ----------------------------------------------------
        # CREATE NEW POPULATION
        # ----------------------------------------------------

        new_population = []

        while len(new_population) < POP_SIZE:

            # ------------------------------------------------
            # SELECTION
            # ------------------------------------------------

            parent1, parent2 = select(
                population
            )

            # ------------------------------------------------
            # CROSSOVER
            # ------------------------------------------------

            if random.random() < CROSSOVER_RATE:

                child1, child2 = crossover(
                    parent1,
                    parent2
                )

            else:

                child1 = [
                    list(parent1[0]),
                    list(parent1[1])
                ]

                child2 = [
                    list(parent2[0]),
                    list(parent2[1])
                ]

            # ------------------------------------------------
            # MUTATION
            # ------------------------------------------------

            mutate(child1)
            mutate(child2)

            # ------------------------------------------------
            # ADD CHILDREN
            # ------------------------------------------------

            new_population.append(
                child1
            )

            if len(new_population) < POP_SIZE:

                new_population.append(
                    child2
                )

        # ----------------------------------------------------
        # REPLACE OLD POPULATION
        # ----------------------------------------------------

        population = new_population

    return best_network, best_fitness


# ============================================================
# RUN GENETIC ALGORITHM
# ============================================================

best_network, best_fitness = genetic_algorithm()

optimal_cluster_heads = best_network[0]
optimal_routes = best_network[1]


# ============================================================
# DISPLAY OPTIMIZATION RESULT
# ============================================================

print("\n=== OPTIMIZATION SUMMARY ===")

print(
    f"Optimal Cluster Heads: "
    f"{optimal_cluster_heads}"
)

print(
    f"Best Fitness Score: "
    f"{best_fitness:.4f}"
)


# ============================================================
# DISPLAY NODE COORDINATES
# ============================================================

print("\n=== NODE COORDINATES ===")

for node in range(NUM_NODES):

    print(
        f"Node {node}: "
        f"({node_x[node]:.2f}, "
        f"{node_y[node]:.2f})"
    )


# ============================================================
# DISPLAY OPTIMAL ROUTING MAP
# ============================================================

print("\n=== OPTIMAL ROUTING MAP ===")

for node in range(NUM_NODES):

    # Cluster Head
    if node in optimal_cluster_heads:

        print(
            f"Node {node:2d} "
            f"is designated as [CLUSTER HEAD]"
        )

    # Normal wireless node
    else:

        assigned_cluster_head = (
            optimal_routes[node]
        )

        distance = calculate_distance(
            node,
            assigned_cluster_head
        )

        print(
            f"Node {node:2d} connects to ---> "
            f"Cluster Head "
            f"{assigned_cluster_head:2d} "
            f"| Distance: "
            f"{distance:.2f}m"
        )


# ============================================================
# VERIFY ROUTES
# ============================================================

print("\n=== ROUTE VALIDATION ===")

valid = True

for node in range(NUM_NODES):

    if node not in optimal_cluster_heads:

        assigned_head = optimal_routes[node]

        if assigned_head not in optimal_cluster_heads:

            print(
                f"ERROR: Node {node} "
                f"connects to non-Cluster Head "
                f"{assigned_head}"
            )

            valid = False


if valid:

    print(
        "All nodes are connected to valid "
        "Cluster Heads."
    )


=== OPTIMIZATION SUMMARY ===
Optimal Cluster Heads: [5, 8]
Best Fitness Score: 227.2265

=== NODE COORDINATES ===
Node 0: (37.45, 2.06)
Node 1: (95.07, 96.99)
Node 2: (73.20, 83.24)
Node 3: (59.87, 21.23)
Node 4: (15.60, 18.18)
Node 5: (15.60, 18.34)
Node 6: (5.81, 30.42)
Node 7: (86.62, 52.48)
Node 8: (60.11, 43.19)
Node 9: (70.81, 29.12)

=== OPTIMAL ROUTING MAP ===
Node  0 connects to ---> Cluster Head  5 | Distance: 27.25m
Node  1 connects to ---> Cluster Head  8 | Distance: 64.16m
Node  2 connects to ---> Cluster Head  8 | Distance: 42.13m
Node  3 connects to ---> Cluster Head  8 | Distance: 21.96m
Node  4 connects to ---> Cluster Head  5 | Distance: 0.16m
Node  5 is designated as [CLUSTER HEAD]
Node  6 connects to ---> Cluster Head  5 | Distance: 15.55m
Node  7 connects to ---> Cluster Head  8 | Distance: 28.08m
Node  8 is designated as [CLUSTER HEAD]
Node  9 connects to ---> Cluster Head  8 | Distance: 17.68m

=== ROUTE VALIDATION ===
All nodes are connected to valid Cluster He